# The Logistic Map
### Lesson 5, Section 1

In this section you will explore the **logistic map** — one of the simplest
equations that can produce chaos:

$$X_{N+1} = r \, X_N (1 - X_N)$$

You will implement the map step by step, visualise trajectories, cobweb
diagrams, and the full bifurcation diagram, and investigate sensitivity
to initial conditions.


## 0 · Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('All imports OK ✓')


---
# Part 1 — The Logistic Map

The **logistic map** is one of the simplest equations that can produce chaos:

$$X_{N+1} = r \, X_N (1 - X_N)$$

- $X_N \in [0, 1]$ represents the normalised population size at generation $N$.
- $r \in [0, 4]$ is the growth-rate parameter.

Despite its simplicity, as $r$ increases the system goes through:
- **stable fixed points** → **period-2 oscillations** → **period-4** → **period-8** → … → **chaos**

This is called a **period-doubling cascade**.


## 1.1 · One iteration of the logistic map
We start with the smallest possible building block: computing a single $X_{N+1}$ from $X_N$.


In [ ]:
def logistic_step(x, r):
    """Compute one step of the logistic map: x_{n+1} = r * x_n * (1 - x_n)."""
    return r * x * (1 - x)

print(logistic_step(0.5, r=2.0))


### Independent test 1
If $X_N = 0$ or $X_N = 1$, the next value should always be $0$ regardless of $r$.


In [ ]:
# --- Test: boundary values ---
result_zero = logistic_step(0.0, r=3.5)
result_one  = logistic_step(1.0, r=3.5)
print(f'logistic_step(0.0, 3.5) = {result_zero}')
print(f'logistic_step(1.0, 3.5) = {result_one}')
assert result_zero == 0.0
assert result_one  == 0.0
print('Test passed ✓ Boundary values behave correctly.')


## 1.2 · Iterating the logistic map
Now we iterate the map many times and record the full trajectory $X_0, X_1, \ldots, X_N$.


In [ ]:
def logistic_trajectory(x0, r, n_steps):
    """Return an array of length n_steps+1 with the full trajectory."""
    x = np.zeros(n_steps + 1)
    x[0] = x0
    for i in range(n_steps):
        x[i + 1] = logistic_step(x[i], r)
    return x

traj = logistic_trajectory(x0=0.5, r=2.5, n_steps=30)
print(f'First 6 values: {traj[:6].round(4)}')
print(f'Last value:      {traj[-1]:.6f}')


### Independent test 2
For $r = 2.5$ the fixed point is $X^* = 1 - 1/r = 0.6$. After many steps the trajectory should converge there.


In [ ]:
# --- Test: convergence to fixed point ---
x_star = 1 - 1 / 2.5
traj_test = logistic_trajectory(0.2, r=2.5, n_steps=100)
print(f'Expected fixed point: {x_star:.4f}')
print(f'Trajectory end value: {traj_test[-1]:.6f}')
assert np.isclose(traj_test[-1], x_star, atol=1e-6)
print('Test passed ✓ Trajectory converges to the fixed point.')


## 1.3 · Visualising different regimes
Let's see how the behaviour changes as $r$ increases.


In [ ]:
r_values = [1.5, 2.8, 3.3, 3.5, 3.56, 3.9]
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

for ax, r_val in zip(axes.flat, r_values):
    traj = logistic_trajectory(0.5, r_val, 60)
    ax.plot(traj, 'o-', markersize=3, linewidth=1, color='steelblue')
    ax.set_title(f'r = {r_val}')
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('$X_N$')

axes[-1, 0].set_xlabel('Generation N')
axes[-1, 1].set_xlabel('Generation N')
plt.suptitle('Logistic Map — Time Series for Different r', fontweight='bold')
plt.tight_layout()
plt.show()


## 1.4 · The cobweb diagram
A **cobweb diagram** shows the iteration graphically:
- Plot the parabola $y = r\,x(1-x)$ and the diagonal $y = x$.
- Starting from $X_0$, go **vertically** to the parabola, then **horizontally** to the diagonal. Repeat.


In [ ]:
def plot_cobweb(x0, r, n_steps=40, ax=None):
    """Draw a cobweb diagram for the logistic map."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    x_curve = np.linspace(0, 1, 300)
    y_curve = r * x_curve * (1 - x_curve)

    ax.plot(x_curve, y_curve, 'b-', linewidth=2, label=f'$f(x) = {r}x(1-x)$')
    ax.plot([0, 1], [0, 1], 'k-', linewidth=1, label='$y = x$')

    x = x0
    ax.plot([x, x], [0, logistic_step(x, r)], 'r-', linewidth=0.8)
    for _ in range(n_steps):
        x_new = logistic_step(x, r)
        ax.plot([x, x_new], [x_new, x_new], 'r-', linewidth=0.8)
        ax.plot([x_new, x_new], [x_new, logistic_step(x_new, r)], 'r-', linewidth=0.8)
        x = x_new

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('$X_N$')
    ax.set_ylabel('$X_{N+1}$')
    ax.set_title(f'Cobweb · r = {r}')
    ax.legend(fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, r_val in zip(axes, [1.5, 3.2, 3.9]):
    plot_cobweb(0.2, r_val, ax=ax)
plt.tight_layout()
plt.show()


## 1.5 · The bifurcation diagram
The **bifurcation diagram** shows, for each $r$, the long-term values that $X_N$ visits. It reveals the period-doubling route to chaos at a glance.


In [ ]:
def bifurcation_diagram(r_min=2.5, r_max=4.0, n_r=4000, n_transient=500, n_plot=200):
    """Compute (r, x) pairs for a bifurcation diagram."""
    r_values = np.linspace(r_min, r_max, n_r)
    x = 0.5 * np.ones_like(r_values)

    # discard transients
    for _ in range(n_transient):
        x = r_values * x * (1 - x)

    rs, xs = [], []
    for _ in range(n_plot):
        x = r_values * x * (1 - x)
        rs.append(r_values.copy())
        xs.append(x.copy())

    return np.concatenate(rs), np.concatenate(xs)

r_pts, x_pts = bifurcation_diagram()

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(r_pts, x_pts, ',', color='black', markersize=0.02, alpha=0.5)
ax.set_xlabel('r')
ax.set_ylabel('$X_N$ (long-term)')
ax.set_title('Bifurcation Diagram of the Logistic Map')
ax.set_xlim(2.5, 4.0)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 1.6 · Sensitivity to initial conditions
In the chaotic regime, two trajectories starting very close together diverge rapidly.


In [ ]:
r_chaos = 3.9
traj_a = logistic_trajectory(0.5000000, r_chaos, 60)
traj_b = logistic_trajectory(0.5000001, r_chaos, 60)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(traj_a, 'o-', markersize=3, label='$X_0 = 0.5000000$', color='steelblue')
axes[0].plot(traj_b, 'o-', markersize=3, label='$X_0 = 0.5000001$', color='firebrick')
axes[0].set_ylabel('$X_N$')
axes[0].set_title('Two nearby trajectories in the chaotic regime (r = 3.9)')
axes[0].legend()

axes[1].plot(np.abs(traj_a - traj_b), 'o-', markersize=3, color='purple')
axes[1].set_ylabel('$|X_A - X_B|$')
axes[1].set_xlabel('Generation N')
axes[1].set_title('Difference between the two trajectories')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print('Observation: after ~20 generations the tiny initial difference has grown to order 1.')


### What to try
- Change `r_chaos` to values like `3.5` (periodic) and compare the divergence plot.
- Try `r = 4.0` — is chaos stronger or weaker?
- Try different initial separations (e.g. $10^{-12}$) — how many extra steps do you gain?


---
## What to Try

1. Find the value of $r$ where the first period-doubling occurs (from fixed
   point to period-2). Hint: it is near $r = 3$.
2. Find the period-3 window in the bifurcation diagram. At what $r$ does it start?
3. Compute the **Feigenbaum ratio**: measure the $r$-intervals between successive
   doublings and compute their ratio. Does it approach $\delta \approx 4.669$?
4. Change `r_chaos` to `3.5` (periodic) and compare the divergence plot.
5. Try different initial separations (e.g. $10^{-12}$) — how many extra steps do you gain?


---
## Mini-Project · Lyapunov Exponent

Compute the **Lyapunov exponent** $\lambda$ as a function of $r$:

$$\lambda(r) = \lim_{N\to\infty} \frac{1}{N} \sum_{n=0}^{N-1} \ln |f'(X_n)|$$

where $f'(x) = r(1 - 2x)$ for the logistic map. Plot $\lambda(r)$ alongside
the bifurcation diagram. Chaos corresponds to $\lambda > 0$; stable behaviour
to $\lambda < 0$. Do the bifurcation points line up with $\lambda = 0$?
